# Notebook 2: Baseline FP-Growth và lọc hậu xử lý

Notebook này thực hiện kịch bản **Baseline (Post-processing)** cho bài thực nghiệm khai phá luật kết hợp định lượng.

Ý tưởng chính:

1. Dùng danh sách giao dịch đã tạo ở `data_preprocessing.ipynb`.
2. Chạy thuật toán FP-Growth nguyên bản, trong đó item được sắp xếp theo **support giảm dần** khi xây FP-Tree.
3. Sinh toàn bộ frequent itemsets theo từng ngưỡng `min_support`.
4. Sau khi mining xong mới lọc bỏ các itemset có `avg(price) > 20`.

Kết quả của notebook này sẽ được lưu lại để so sánh với thuật toán đề xuất trong các notebook tiếp theo.

## 1. Cấu hình môi trường và thư mục đầu ra

Notebook được thiết kế để chạy được trên cả local và Google Colab. Nếu chạy trên Colab, hãy chạy `data_preprocessing.ipynb` trước hoặc upload thư mục artifact đã sinh từ notebook 1. Có thể đổi thư mục artifact bằng biến môi trường `OUTPUT_DIR`, đổi danh sách support bằng `MIN_SUPPORT_RATIOS`, và đổi ngưỡng giá bằng `PRICE_THRESHOLD`.

In [1]:
import json
import math
import os
import pickle
import sys
from pathlib import Path

PROJECT_ROOT = Path(os.environ.get("PROJECT_ROOT", Path.cwd())).resolve()
root_dir = None
for candidate in [PROJECT_ROOT, *PROJECT_ROOT.parents]:
    if (candidate / "src").exists():
        root_dir = candidate
        break
if root_dir is None:
    root_dir = PROJECT_ROOT
# Ensure repo root for data/output paths when running from notebooks.
PROJECT_ROOT = root_dir
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src import (
    ensure_package,
    find_artifact,
    parse_support_ratios,
    patterns_to_dataframe,
    run_baseline_fp_growth,
 )

ensure_package("pandas")

import pandas as pd

try:
    from IPython.display import display
except Exception:
    def display(obj):
        print(obj)

try:
    import google.colab  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False

# =========================
# CẤU HÌNH LINH HOẠT
# =========================
# Có thể chỉnh bằng biến môi trường hoặc sửa trực tiếp tại đây khi chạy trên Colab/local.
OUTPUT_DIR = Path(os.environ.get("OUTPUT_DIR", PROJECT_ROOT / "outputs")).resolve()
DATA_DIR = Path(os.environ.get("DATA_DIR", PROJECT_ROOT / "data")).resolve()

ARTIFACT_FILES = {
    "transactions_pickle": os.environ.get("TRANSACTIONS_PICKLE", "transactions.pkl"),
    "item_price_pickle": os.environ.get("ITEM_PRICE_PICKLE", "item_price.pkl"),
    "preprocessing_summary_json": os.environ.get("PREPROCESSING_SUMMARY_JSON", "preprocessing_summary.json"),
    "baseline_results_pickle": os.environ.get("BASELINE_RESULTS_PICKLE", "baseline_results.pkl"),
    "baseline_metrics_csv": os.environ.get("BASELINE_METRICS_CSV", "baseline_metrics.csv"),
    "baseline_patterns_csv": os.environ.get("BASELINE_PATTERNS_CSV", "baseline_final_patterns.csv"),
    "baseline_summary_json": os.environ.get("BASELINE_SUMMARY_JSON", "baseline_summary.json"),
}

MIN_SUPPORT_RATIOS = parse_support_ratios(
    os.environ.get("MIN_SUPPORT_RATIOS", ""),
    default_ratios=[0.05, 0.04, 0.03, 0.02, 0.01],
)
PRICE_THRESHOLD = float(os.environ.get("PRICE_THRESHOLD", "20"))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 180)
pd.set_option("display.max_colwidth", 180)
pd.set_option("display.max_rows", 3000)

print("Môi trường chạy:", "Google Colab" if IN_COLAB else "Local/Jupyter")
print("Thư mục làm việc:", PROJECT_ROOT)
print("Thư mục dữ liệu:", DATA_DIR)
print("Thư mục lưu kết quả:", OUTPUT_DIR)
print("Danh sách min_support:", [f"{ratio:.0%}" for ratio in MIN_SUPPORT_RATIOS])
print("Ngưỡng avg(price):", PRICE_THRESHOLD)
if not DATA_DIR.exists():
    print("Warning: DATA_DIR not found:", DATA_DIR)

Môi trường chạy: Local/Jupyter
Thư mục làm việc: D:\Projects\Constraint-based_pattern_data_mining
Thư mục dữ liệu: D:\Projects\Constraint-based_pattern_data_mining\data
Thư mục lưu kết quả: D:\Projects\Constraint-based_pattern_data_mining\outputs
Danh sách min_support: ['5%', '4%', '3%', '2%', '1%']
Ngưỡng avg(price): 20.0


## 2. Nạp dữ liệu tiền xử lý

Notebook dùng các artifact đã được tạo bởi `data_preprocessing.ipynb`:

- File giao dịch dạng pickle, mặc định `transactions.pkl`.
- File giá đại diện của từng item, mặc định `item_price.pkl`.
- File số liệu tổng hợp tiền xử lý, mặc định `preprocessing_summary.json`.

Các tên file mặc định có thể đổi trong cell cấu hình hoặc bằng biến môi trường.

In [2]:
transactions_path = find_artifact(
    ARTIFACT_FILES["transactions_pickle"],
    output_dir=OUTPUT_DIR,
    project_root=PROJECT_ROOT,
    in_colab=IN_COLAB,
)
item_price_path = find_artifact(
    ARTIFACT_FILES["item_price_pickle"],
    output_dir=OUTPUT_DIR,
    project_root=PROJECT_ROOT,
    in_colab=IN_COLAB,
)
summary_path = find_artifact(
    ARTIFACT_FILES["preprocessing_summary_json"],
    output_dir=OUTPUT_DIR,
    project_root=PROJECT_ROOT,
    in_colab=IN_COLAB,
)

required_artifacts = [transactions_path, item_price_path, summary_path]
missing_artifacts = [str(path) for path in required_artifacts if not path.exists()]
if missing_artifacts:
    raise FileNotFoundError(
        "Thiếu artifact từ bước tiền xử lý. Hãy chạy data_preprocessing.ipynb trước. "
        f"Các file còn thiếu: {missing_artifacts}"
    )

with transactions_path.open("rb") as f:
    transactions = pickle.load(f)

with item_price_path.open("rb") as f:
    item_price = pickle.load(f)

with summary_path.open("r", encoding="utf-8") as f:
    preprocessing_summary = json.load(f)

load_summary_df = pd.DataFrame([
    {"Thông tin": "Số giao dịch", "Giá trị": len(transactions)},
    {"Thông tin": "Số item có giá đại diện", "Giá trị": len(item_price)},
    {"Thông tin": "Số dòng sau tiền xử lý", "Giá trị": preprocessing_summary.get("clean_rows")},
    {"Thông tin": "Số mặt hàng duy nhất", "Giá trị": preprocessing_summary.get("unique_items")},
])

print("Đã nạp dữ liệu tiền xử lý thành công.")
display(load_summary_df)

print("Ví dụ 5 giao dịch đầu:")
sample_transactions_df = pd.DataFrame({
    "Mã thứ tự giao dịch": list(range(1, 6)),
    "Số item": [len(t) for t in transactions[:5]],
    "Items": [" | ".join(t) for t in transactions[:5]],
})
display(sample_transactions_df)

Đã nạp dữ liệu tiền xử lý thành công.


,Thông tin,Giá trị
0,Số giao dịch,19559
1,Số item có giá đại diện,4006
2,Số dòng sau tiền xử lý,519551
3,Số mặt hàng duy nhất,4006


Ví dụ 5 giao dịch đầu:


,Mã thứ tự giao dịch,Số item,Items
0,1,7,CREAM CUPID HEARTS COAT HANGER | GLASS STAR FROSTED T-LIGHT HOLDER | KNITTED UNION FLAG HOT WATER BOTTLE | RED WOOLLY HOTTIE WHITE HEART. | SET 7 BABUSHKA NESTING BOXES | WHITE...
1,2,2,HAND WARMER RED POLKA DOT | HAND WARMER UNION JACK
2,3,12,ASSORTED COLOUR BIRD ORNAMENT | BOX OF 6 ASSORTED COLOUR TEASPOONS | BOX OF VINTAGE ALPHABET BLOCKS | BOX OF VINTAGE JIGSAW BLOCKS | DOORMAT NEW ENGLAND | FELTCRAFT PRINCESS CH...
3,4,4,BLUE COAT RACK PARIS FASHION | JAM MAKING SET WITH JARS | RED COAT RACK PARIS FASHION | YELLOW COAT RACK PARIS FASHION
4,5,1,BATH BUILDING BLOCK WORD


## 3. Cấu hình thực nghiệm Baseline

Các ngưỡng `min_support` được đọc từ cấu hình `MIN_SUPPORT_RATIOS`, mặc định là `5%`, `4%`, `3%`, `2%`, `1%`. Với mỗi ngưỡng, notebook đổi sang số lượng giao dịch tối thiểu bằng công thức:

`min_support_count = ceil(min_support_ratio * số_giao_dịch)`

Điều kiện định lượng dùng ở bước hậu xử lý là `avg(price) <= PRICE_THRESHOLD`, mặc định `20`.

In [3]:
N_TRANSACTIONS = len(transactions)

support_config_df = pd.DataFrame([
    {
        "Min Support": f"{ratio:.0%}",
        "Tỷ lệ": ratio,
        "Support count tối thiểu": math.ceil(ratio * N_TRANSACTIONS),
    }
    for ratio in MIN_SUPPORT_RATIOS
])

print("Cấu hình thực nghiệm:")
display(support_config_df)
print(f"Ngưỡng lọc hậu xử lý: avg(price) <= {PRICE_THRESHOLD}")

Cấu hình thực nghiệm:


,Min Support,Tỷ lệ,Support count tối thiểu
0,5%,0.05,978
1,4%,0.04,783
2,3%,0.03,587
3,2%,0.02,392
4,1%,0.01,196


Ngưỡng lọc hậu xử lý: avg(price) <= 20.0


## 4. Cài đặt FP-Growth Baseline

Phần này cài đặt FP-Growth theo đúng tinh thần baseline:

- Tính support của item đơn.
- Giữ các item đạt `min_support`.
- Sắp xếp item trong mỗi giao dịch theo **support giảm dần** khi xây FP-Tree.
- Sinh toàn bộ frequent itemsets bằng pattern growth.
- Chỉ sau khi sinh xong mới lọc theo điều kiện `avg(price) <= 20`.

Cài đặt có thêm tối ưu single-path để tăng tốc khi cây điều kiện chỉ còn một đường đi.

In [4]:
print("Đã định nghĩa xong cấu trúc FP-Tree và hàm chạy Baseline FP-Growth.")

Đã định nghĩa xong cấu trúc FP-Tree và hàm chạy Baseline FP-Growth.


## 5. Chạy thực nghiệm Baseline
Với mỗi ngưỡng support, notebook đo thời gian từ lúc xây FP-Tree đến khi hoàn tất lọc hậu xử lý. Đây là thời gian dùng để so sánh với thuật toán đề xuất.

In [5]:
baseline_results = {}
metrics_rows = []

for ratio in MIN_SUPPORT_RATIOS:
    label = f"{ratio:.0%}"
    print(f"Đang chạy Baseline FP-Growth với min_support = {label}...")
    result = run_baseline_fp_growth(
        transactions=transactions,
        item_price=item_price,
        min_support_ratio=ratio,
        price_threshold=PRICE_THRESHOLD,
    )
    baseline_results[label] = result

    metrics_rows.append({
        "Min Support": label,
        "Support count tối thiểu": result["min_support_count"],
        "Thời gian Baseline (giây)": round(result["runtime_seconds"], 6),
        "Số frequent itemsets trước lọc": result["num_all_patterns"],
        "Số itemsets bị lọc do avg(price) > 20": result["num_filtered_patterns"],
        "Số lượng Final Patterns": result["num_final_patterns"],
    })
    print(
        f"Hoàn tất {label}: "
        f"{result['num_all_patterns']} frequent itemsets, "
        f"{result['num_final_patterns']} final patterns, "
        f"runtime = {result['runtime_seconds']:.4f} giây."
    )

baseline_metrics_df = pd.DataFrame(metrics_rows)

print("Bảng kết quả Baseline:")
display(baseline_metrics_df)

Đang chạy Baseline FP-Growth với min_support = 5%...
Hoàn tất 5%: 33 frequent itemsets, 33 final patterns, runtime = 0.2422 giây.
Đang chạy Baseline FP-Growth với min_support = 4%...
Hoàn tất 4%: 66 frequent itemsets, 66 final patterns, runtime = 0.3384 giây.
Đang chạy Baseline FP-Growth với min_support = 3%...
Hoàn tất 3%: 138 frequent itemsets, 137 final patterns, runtime = 0.5587 giây.
Đang chạy Baseline FP-Growth với min_support = 2%...
Hoàn tất 2%: 382 frequent itemsets, 379 final patterns, runtime = 1.4234 giây.
Đang chạy Baseline FP-Growth với min_support = 1%...
Hoàn tất 1%: 1948 frequent itemsets, 1862 final patterns, runtime = 4.2257 giây.
Bảng kết quả Baseline:


,Min Support,Support count tối thiểu,Thời gian Baseline (giây),Số frequent itemsets trước lọc,Số itemsets bị lọc do avg(price) > 20,Số lượng Final Patterns
0,5%,978,0.242170,33,0,33
1,4%,783,0.338368,66,0,66
2,3%,587,0.558719,138,1,137
3,2%,392,1.423404,382,3,379
4,1%,196,4.225731,1948,86,1862


## 6. Hiển thị đầy đủ các final patterns của Baseline
Bảng dưới đây hiển thị các itemset sau hậu xử lý với avg(price) <= 20. Toàn bộ bảng cũng được lưu thành CSV để tiện kiểm tra lại.

In [6]:
all_final_patterns_df = pd.concat(
    [patterns_to_dataframe(result, item_price, N_TRANSACTIONS) for result in baseline_results.values()],
    ignore_index=True,
)

all_final_patterns_df["Support"] = all_final_patterns_df["Support"].round(6)
all_final_patterns_df["Avg Price"] = all_final_patterns_df["Avg Price"].round(6)

# print("Toàn bộ final patterns của Baseline sau khi lọc hậu xử lý:")
# display(all_final_patterns_df)

## 7. Lưu kết quả Baseline
Các kết quả được lưu để notebook đánh giá có thể so sánh trực tiếp với thuật toán đề xuất.

In [7]:
baseline_results_path = OUTPUT_DIR / ARTIFACT_FILES["baseline_results_pickle"]
baseline_metrics_path = OUTPUT_DIR / ARTIFACT_FILES["baseline_metrics_csv"]
baseline_patterns_path = OUTPUT_DIR / ARTIFACT_FILES["baseline_patterns_csv"]
baseline_summary_path = OUTPUT_DIR / ARTIFACT_FILES["baseline_summary_json"]

with baseline_results_path.open("wb") as f:
    pickle.dump(baseline_results, f)

baseline_metrics_df.to_csv(baseline_metrics_path, index=False, encoding="utf-8-sig")
all_final_patterns_df.to_csv(baseline_patterns_path, index=False, encoding="utf-8-sig")

baseline_summary = {
    "price_threshold": PRICE_THRESHOLD,
    "num_transactions": N_TRANSACTIONS,
    "min_supports": [f"{ratio:.0%}" for ratio in MIN_SUPPORT_RATIOS],
    "metrics": baseline_metrics_df.to_dict(orient="records"),
}
with baseline_summary_path.open("w", encoding="utf-8") as f:
    json.dump(baseline_summary, f, ensure_ascii=False, indent=2)

saved_files_df = pd.DataFrame([
    {"File": str(baseline_results_path), "Mô tả": "Kết quả Baseline dạng pickle để evaluation/proposed dùng so sánh"},
    {"File": str(baseline_metrics_path), "Mô tả": "Bảng runtime và số lượng pattern của Baseline"},
    {"File": str(baseline_patterns_path), "Mô tả": "Toàn bộ final patterns của Baseline sau hậu xử lý"},
    {"File": str(baseline_summary_path), "Mô tả": "Tóm tắt kết quả Baseline dạng JSON"},
])

print("Đã lưu xong kết quả Baseline:")
display(saved_files_df)

Đã lưu xong kết quả Baseline:


,File,Mô tả
0,D:\Projects\Constraint-based_pattern_data_mining\outputs\baseline_results.pkl,Kết quả Baseline dạng pickle để evaluation/proposed dùng so sánh
1,D:\Projects\Constraint-based_pattern_data_mining\outputs\baseline_metrics.csv,Bảng runtime và số lượng pattern của Baseline
2,D:\Projects\Constraint-based_pattern_data_mining\outputs\baseline_final_patterns.csv,Toàn bộ final patterns của Baseline sau hậu xử lý
3,D:\Projects\Constraint-based_pattern_data_mining\outputs\baseline_summary.json,Tóm tắt kết quả Baseline dạng JSON


## 8. Nhận xét nhanh cho kịch bản Baseline
Baseline chạy FP-Growth theo cách chuẩn, ưu tiên nén FP-Tree bằng cách sắp xếp item theo support giảm dần. Tuy nhiên, do ràng buộc định lượng avg(price) <= 20 chỉ được áp dụng sau khi sinh xong frequent itemsets, thuật toán vẫn phải tạo cả những mẫu cuối cùng sẽ bị loại bỏ. Đây là điểm sẽ được so sánh với hướng In-mining Pruning ở notebook tiếp theo.